In [0]:
%run "../00_common/01_environment_config"

In [0]:
# Databricks notebook source

# ============================================================
# F1 PROJECT - TRANSFORM RESULTS
# ============================================================
# Purpose:
# Cleans and transforms Bronze results data and writes the
# processed data in Delta format to the Silver layer.
# ============================================================


# COMMAND ----------

# MAGIC %run "../00_common/01_environment_config"


# COMMAND ----------

from pyspark.sql import functions as F


# COMMAND ----------

# ============================================================
# SOURCE AND TARGET PATHS
# ============================================================

bronze_path = f"{s3_root_path}/Bronze/Kalashri/sprints"
silver_path = f"{s3_root_path}/Silver/Kalashri/sprints"

print("Bronze Path :", bronze_path)
print("Silver Path :", silver_path)


# COMMAND ----------

# ============================================================
# STEP 1 - READ BRONZE RESULTS DATA
# ============================================================

results_bronze_df = (
    spark.read
        .format("delta")
        .load(bronze_path)
)

display(results_bronze_df.limit(20))


# COMMAND ----------

# ============================================================
# STEP 2 - SELECT REQUIRED COLUMNS AND RENAME
# ============================================================
# URL is excluded because it is not required for analytics.

results_df = (
    results_bronze_df
        .select(
            "season",
            "round",
            "constructorId",
            "driverId",
            "date",
            "raceName",
            "grid",
            "laps",
            "number",
            "points",
            "position",
            "positionText",
            "status",
            "ingestion_timestamp",
            "source_file"
        )
        .withColumnsRenamed({
            "constructorId": "constructor_id",
            "driverId": "driver_id",
            "raceName": "race_name",
            "date": "race_date",
            "grid": "grid_position",
            "laps": "completed_laps",
            "number": "car_number",
            "position": "final_position",
            "positionText": "final_position_text"
        })
)

display(results_df.limit(20))


# COMMAND ----------

# ============================================================
# STEP 3 - APPLY DATA QUALITY CHECKS
# ============================================================
# Remove records where important business keys are NULL.

results_valid_df = (
    results_df
        .filter(
            F.col("season").isNotNull() &
            F.col("round").isNotNull() &
            F.col("constructor_id").isNotNull() &
            F.col("driver_id").isNotNull()
        )
)

display(results_valid_df.limit(20))


# COMMAND ----------

# ============================================================
# STEP 4 - REMOVE DUPLICATES
# ============================================================
# One driver should have one result for a constructor
# in a particular race.

results_distinct_df = (
    results_valid_df
        .dropDuplicates([
            "season",
            "round",
            "constructor_id",
            "driver_id"
        ])
)

display(results_distinct_df.limit(20))


# COMMAND ----------

# ============================================================
# STEP 5 - STANDARDIZE TEXT VALUES
# ============================================================

results_clean_df = (
    results_distinct_df
        .withColumn(
            "race_name",
            F.initcap(F.trim(F.col("race_name")))
        )
)

display(results_clean_df.limit(20))


# COMMAND ----------

# ============================================================
# STEP 6 - CREATE DERIVED COLUMN
# ============================================================
# is_winner = True if the driver finished in position 1.

results_final_df = (
    results_clean_df
        .withColumn(
            "is_winner",
            F.when(
                F.col("final_position") == 1,
                True
            ).otherwise(False)
        )
)

display(results_final_df.limit(20))


# COMMAND ----------

# ============================================================
# STEP 7 - CHECK REMOVED RECORDS
# ============================================================

bronze_count = results_df.count()
silver_count = results_final_df.count()

print("Bronze Records   :", bronze_count)
print("Valid Records    :", silver_count)
print("Records Removed  :", bronze_count - silver_count)


# COMMAND ----------

# ============================================================
# STEP 8 - WRITE TO SILVER
# ============================================================
# overwriteSchema is used because the existing Silver schema
# has changed.

(
    results_final_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(silver_path)
)


# COMMAND ----------

# ============================================================
# STEP 9 - VALIDATE SILVER DATA
# ============================================================

silver_results_df = (
    spark.read
        .format("delta")
        .load(silver_path)
)

display(silver_results_df.limit(20))


# COMMAND ----------

# Check final schema

silver_results_df.printSchema()


# COMMAND ----------

print("Bronze Records :", results_bronze_df.count())
print("Silver Records :", silver_results_df.count())

print("Silver Results data successfully written to:")
print(silver_path)

In [0]:
%sql

SELECT *
FROM delta.`s3://bizf1project123/Silver/Kalashri/sprints`;